# V3 - Target 0.85 in 1 Hour
**Key fix:** Cross-song stem mixing (like test data)

In [1]:
!pip install -q librosa timm

import os, glob, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device: cuda


In [2]:
# Config - Optimized for 0.85 in 1hr
CONFIG = {
    'sr': 22050,
    'duration': 10,
    'n_mels': 128,
    'n_fft': 2048,
    'hop_length': 512,
    'num_classes': 10,
    'batch_size': 32,
    'epochs': 8,
    'lr': 1e-3,
    'noise_prob': 0.8,
    'noise_level': (0.1, 0.4),
    'cross_song_prob': 0.8,  # KEY: Mix stems from different songs
}

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']

BASE_PATH = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_DIR = os.path.join(BASE_PATH, 'genres_stems')
NOISE_DIR = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
SAMPLE_SUB = os.path.join(BASE_PATH, 'sample_submission.csv')

In [3]:
# Load all songs organized by genre
songs_by_genre = {g: [] for g in GENRES}

for genre in GENRES:
    genre_dir = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_dir):
        for song_id in os.listdir(genre_dir):
            song_path = os.path.join(genre_dir, song_id)
            if os.path.isdir(song_path):
                if all(os.path.exists(os.path.join(song_path, f"{s}.wav")) for s in STEMS):
                    songs_by_genre[genre].append(song_path)

for g, songs in songs_by_genre.items():
    print(f"{g}: {len(songs)} songs")

# Noise files
noise_files = glob.glob(os.path.join(NOISE_DIR, '*.wav'))
print(f"\nNoise files: {len(noise_files)}")

blues: 100 songs
classical: 100 songs
country: 100 songs
disco: 100 songs
hiphop: 100 songs
jazz: 100 songs
metal: 100 songs
pop: 100 songs
reggae: 100 songs
rock: 100 songs

Noise files: 2000


In [4]:
# Audio functions
def load_stem(path, sr, duration):
    target_len = sr * duration
    try:
        audio, _ = librosa.load(path, sr=sr, duration=duration)
        if len(audio) < target_len:
            audio = np.pad(audio, (0, target_len - len(audio)))
        return audio[:target_len]
    except:
        return np.zeros(target_len)

def load_noise(sr, duration):
    if not noise_files:
        return np.zeros(sr * duration)
    path = random.choice(noise_files)
    return load_stem(path, sr, duration)

def mix_stems_same_song(song_path, sr, duration):
    """Mix all stems from same song"""
    mixed = np.zeros(sr * duration, dtype=np.float32)
    for stem in STEMS:
        mixed += load_stem(os.path.join(song_path, f"{stem}.wav"), sr, duration)
    return mixed

def mix_stems_cross_song(genre, sr, duration):
    """KEY: Mix stems from DIFFERENT songs (like test data)"""
    mixed = np.zeros(sr * duration, dtype=np.float32)
    genre_songs = songs_by_genre[genre]
    for stem in STEMS:
        song_path = random.choice(genre_songs)
        mixed += load_stem(os.path.join(song_path, f"{stem}.wav"), sr, duration)
    return mixed

def add_noise(audio, level):
    noise = load_noise(CONFIG['sr'], CONFIG['duration'])
    if np.max(np.abs(noise)) > 0:
        noise = noise / np.max(np.abs(noise))
    return audio + level * noise

def normalize(audio):
    audio = audio - np.mean(audio)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.9
    return audio.astype(np.float32)

def to_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=CONFIG['sr'], n_mels=CONFIG['n_mels'],
        n_fft=CONFIG['n_fft'], hop_length=CONFIG['hop_length']
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

In [5]:
# Dataset with cross-song mixing
class CrossSongDataset(Dataset):
    def __init__(self, genres_list, samples_per_genre=100, augment=True):
        self.data = [(g, i) for g in genres_list for i in range(samples_per_genre)]
        self.augment = augment
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        genre, _ = self.data[idx]
        label = genre_to_idx[genre]
        
        # KEY: Cross-song mixing most of the time
        if self.augment and random.random() < CONFIG['cross_song_prob']:
            audio = mix_stems_cross_song(genre, CONFIG['sr'], CONFIG['duration'])
        else:
            song_path = random.choice(songs_by_genre[genre])
            audio = mix_stems_same_song(song_path, CONFIG['sr'], CONFIG['duration'])
        
        # Add noise
        if self.augment and random.random() < CONFIG['noise_prob']:
            level = random.uniform(*CONFIG['noise_level'])
            audio = add_noise(audio, level)
        
        # Time shift
        if self.augment and random.random() < 0.5:
            shift = random.randint(-CONFIG['sr'], CONFIG['sr'])
            audio = np.roll(audio, shift)
        
        audio = normalize(audio)
        mel = to_mel(audio)
        
        # SpecAugment
        if self.augment:
            if random.random() < 0.5:
                t = random.randint(0, 20)
                t0 = random.randint(0, mel.shape[1] - t - 1)
                mel[:, t0:t0+t] = 0
            if random.random() < 0.5:
                f = random.randint(0, 15)
                f0 = random.randint(0, mel.shape[0] - f - 1)
                mel[f0:f0+f, :] = 0
        
        mel_t = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        return mel_t, label

# Test dataset
class TestDataset(Dataset):
    def __init__(self, file_paths):
        self.paths = file_paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        audio = load_stem(self.paths[idx], CONFIG['sr'], CONFIG['duration'])
        audio = normalize(audio)
        mel = to_mel(audio)
        return torch.tensor(mel, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)

In [6]:
# Create datasets
# More samples per genre for better learning
train_dataset = CrossSongDataset(GENRES, samples_per_genre=150, augment=True)
val_dataset = CrossSongDataset(GENRES, samples_per_genre=20, augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

Train samples: 1500
Val samples: 200
Batches per epoch: 47


In [7]:
# Model - EfficientNet-B0
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.backbone.num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CONFIG['num_classes'])
        )
    def forward(self, x):
        return self.head(self.backbone(x))

model = Model().to(device)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Params: 4,338,054


In [8]:
# Training
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['lr'], epochs=CONFIG['epochs'], steps_per_epoch=len(train_loader)
)

best_acc = 0
for epoch in range(CONFIG['epochs']):
    model.train()
    train_correct, train_total = 0, 0
    
    for data, target in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_correct += (output.argmax(1) == target).sum().item()
        train_total += target.size(0)
    
    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            val_correct += (model(data).argmax(1) == target).sum().item()
            val_total += target.size(0)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best.pth')
    
    print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}, Best={best_acc:.4f}")

print(f"\nBest: {best_acc:.4f}")

Epoch 1: 100%|██████████| 47/47 [03:47<00:00,  4.84s/it]


Epoch 1: Train=0.2600, Val=0.5500, Best=0.5500


Epoch 2: 100%|██████████| 47/47 [02:59<00:00,  3.81s/it]


Epoch 2: Train=0.6553, Val=0.6750, Best=0.6750


Epoch 3: 100%|██████████| 47/47 [02:51<00:00,  3.64s/it]


...

Epoch 6: Train=0.8680, Val=0.9450, Best=0.9450


Epoch 7: 100%|██████████| 47/47 [02:47<00:00,  3.57s/it]


Epoch 7: Train=0.8853, Val=0.9200, Best=0.9450


Epoch 8: 100%|██████████| 47/47 [02:47<00:00,  3.56s/it]


Epoch 8: Train=0.9007, Val=0.9250, Best=0.9450

Best: 0.9450


In [9]:
# Test inference
model.load_state_dict(torch.load('best.pth'))
model.eval()

test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)
id_to_file = dict(zip(test_df['id'], test_df['filename']))

test_files = [os.path.join(BASE_PATH, id_to_file[row['id']]) for _, row in sample_sub.iterrows()]
print(f"Test files: {len(test_files)}")

test_loader = DataLoader(TestDataset(test_files), batch_size=CONFIG['batch_size'], num_workers=2)

preds = []
with torch.no_grad():
    for data in tqdm(test_loader, desc="Testing"):
        preds.extend(model(data.to(device)).argmax(1).cpu().numpy())

submission = sample_sub.copy()
submission['genre'] = [GENRES[p] for p in preds]
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")
print(submission['genre'].value_counts())

Test files: 3020


Testing: 100%|██████████| 95/95 [02:05<00:00,  1.32s/it]


Submission saved!
genre
pop          452
hiphop       364
rock         344
reggae       339
metal        306
jazz         301
disco        262
classical    245
blues        237
country      170
Name: count, dtype: int64
